In [1]:
!pip install medmnist scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 4.8 MB/s eta 0:00:00


In [2]:
import medmnist
from medmnist import PathMNIST
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
epochs = 5
lr = 0.001
num_classes = 9

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ]),
    'val/test': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5])
    ])
}

In [7]:
import os

data_root = "./medmnist_data"
os.makedirs(data_root, exist_ok=True)

train_dataset = PathMNIST(split='train', transform=data_transforms['train'], download=True, root=data_root)
val_dataset = PathMNIST(split='val', transform=data_transforms['val/test'], download=True, root=data_root)
test_dataset = PathMNIST(split='test', transform=data_transforms['val/test'], download=True, root=data_root)

In [8]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [13]:
from torchvision.models import resnet18
model1 = resnet18(weights=None)
model1.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
model1.fc = nn.Linear(model1.fc.in_features, num_classes)
model1 = model1.to(device)

from torchvision.models import mobilenet_v2
model2 = mobilenet_v2(weights=None)
model2.features[0][0] = nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1, bias=False)
model2.classifier[1] = nn.Linear(model2.classifier[1].in_features, num_classes)
model2 = model2.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters(), lr=lr)

In [10]:
print("开始训练 ResNet18...")
for epoch in range(epochs):
    model1.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.squeeze().long().to(device)
        optimizer.zero_grad()
        outputs = model1(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    
    model1.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.squeeze().long().to(device)
            outputs = model1(inputs)
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    train_loss = train_loss / len(train_loader.dataset)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}")

print("\n开始测试 ResNet18...")
model1.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.squeeze().long().to(device)
        outputs = model1(inputs)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

print("\nResNet18 测试结果：")
print(f"准确率: {accuracy_score(test_labels, test_preds):.4f}")
print("\n分类报告:")
print(classification_report(test_labels, test_preds))
print("\n混淆矩阵:")
print(confusion_matrix(test_labels, test_preds))

开始训练 ResNet18...
Epoch 1/5, Train Loss: 0.7840, Val Acc: 0.7776
Epoch 2/5, Train Loss: 0.5169, Val Acc: 0.6450
Epoch 3/5, Train Loss: 0.3976, Val Acc: 0.6537
Epoch 4/5, Train Loss: 0.3430, Val Acc: 0.6997
Epoch 5/5, Train Loss: 0.2982, Val Acc: 0.8232

开始测试 ResNet18...

ResNet18 测试结果：
准确率: 0.6320

分类报告:
              precision    recall  f1-score   support

           0       0.65      0.19      0.29      1338
           1       0.93      1.00      0.96       847
           2       0.29      0.76      0.42       339
           3       0.91      0.94      0.93       634
           4       0.98      0.54      0.70      1035
           5       0.23      0.64      0.34       592
           6       0.86      0.62      0.72       741
           7       0.36      0.32      0.34       421
           8       0.87      0.85      0.86      1233

    accuracy                           0.63      7180
   macro avg       0.68      0.65      0.62      7180
weighted avg       0.75      0.63      0.64  

In [15]:
print("开始训练 mobilenet...")
for epoch in range(epochs):
    model2.train()
    train_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.squeeze().long().to(device)
        optimizer.zero_grad()
        outputs = model1(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
    
    model2.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.squeeze().long().to(device)
            outputs = model1(inputs)
            _, preds = torch.max(outputs, 1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    train_loss = train_loss / len(train_loader.dataset)
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}")

print("\n开始测试 mobilenet...")
model2.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.squeeze().long().to(device)
        outputs = model1(inputs)
        _, preds = torch.max(outputs, 1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())

print("\nmobilenet 测试结果：")
print(f"准确率: {accuracy_score(test_labels, test_preds):.4f}")
print("\n分类报告:")
print(classification_report(test_labels, test_preds))
print("\n混淆矩阵:")
print(confusion_matrix(test_labels, test_preds))

开始训练 mobilenet...
Epoch 1/5, Train Loss: 0.7797, Val Acc: 0.7827
Epoch 2/5, Train Loss: 0.5153, Val Acc: 0.8415
Epoch 3/5, Train Loss: 0.4091, Val Acc: 0.8716
Epoch 4/5, Train Loss: 0.3494, Val Acc: 0.8728
Epoch 5/5, Train Loss: 0.3048, Val Acc: 0.9010

开始测试 mobilenet...

mobilenet 测试结果：
准确率: 0.7670

分类报告:
              precision    recall  f1-score   support

           0       0.98      0.94      0.96      1338
           1       0.90      1.00      0.95       847
           2       0.28      0.60      0.38       339
           3       0.91      0.74      0.81       634
           4       0.84      0.68      0.75      1035
           5       0.69      0.62      0.65       592
           6       0.91      0.57      0.70       741
           7       0.34      0.44      0.39       421
           8       0.79      0.85      0.82      1233

    accuracy                           0.77      7180
   macro avg       0.74      0.72      0.71      7180
weighted avg       0.81      0.77      0.7